# Model training and evaluation


In [1]:
%pip install streamlit
%pip install -q scikit-learn sentence-transformers tqdm
%pip install pandas numpy


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Import necessary dependencies


In [8]:
import json, os
from collections import Counter

import joblib
import numpy as np
import pandas as pd

from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import FeatureUnion
from sentence_transformers import SentenceTransformer


## Define Constants


In [3]:
CONFIG = json.load(open("config.json"))
output_dir = CONFIG["output_dir"]

TOP_K = [5, 10]
MAX_EVAL_QUERIES = 500
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

os.makedirs(output_dir, exist_ok=True)


## Load the clean dataset


In [4]:
questions_df = pd.read_csv(CONFIG["clean_path"])
print(f"Shape of dataset:", questions_df.shape)

Shape of dataset: (40109, 12)


### Data types of dataset

In [5]:
print(questions_df.dtypes)

question_id            int64
title                    str
body_text                str
code_text                str
tags                     str
score                  int64
view_count             int64
answer_count           int64
is_answered             bool
has_accepted_answer     bool
creation_date            str
text_all                 str
dtype: object


### Double check if null value exists

In [6]:
print(questions_df.isnull().sum())

question_id               0
title                     0
body_text                 0
code_text              7569
tags                      0
score                     0
view_count                0
answer_count              0
is_answered               0
has_accepted_answer       0
creation_date             0
text_all                  0
dtype: int64


In [ ]:
# questions_df["question_id"] = questions_df["question_id"].astype(str)
# questions_df["creation_date"] = pd.to_datetime(questions_df["creation_date"], utc=True)
#
# for column_name in ["is_answered", "has_accepted_answer"]:
#     questions_df[column_name] = questions_df[column_name].astype(str).str.lower().map({"true": True, "false": False})
#
# for column_name in ["score", "view_count", "answer_count"]:
#     questions_df[column_name] = pd.to_numeric(questions_df[column_name], errors="coerce").fillna(0).astype(int)

## Prepare labels from the domain tags


In [9]:
domain_tag_set = set(CONFIG["domain_tags"])
questions_df["domain_tag_list"] = questions_df["tags"].str.split("|").map(
    lambda tags: [tag for tag in tags if tag in domain_tag_set])

tag_counts = Counter(tag for tags in questions_df["domain_tag_list"] for tag in tags)
tag_vocabulary = sorted(tag_counts)
tag_to_index = {tag: index for index, tag in enumerate(tag_vocabulary)}

label_matrix = MultiLabelBinarizer(classes=tag_vocabulary).fit_transform(questions_df["domain_tag_list"]).astype(np.int8)
print(f"tag vocabulary: {len(tag_vocabulary)} tags, label matrix: {label_matrix.shape}")

tag vocabulary: 179 tags, label matrix: (40109, 179)


## Split the dataset into train, validation and test sets


In [12]:
RANDOM_SPLIT_SEED = 24
train_df, remaining_df = train_test_split(questions_df, test_size=0.2, random_state=RANDOM_SPLIT_SEED)
val_df, test_df = train_test_split(remaining_df, test_size=0.5, random_state=RANDOM_SPLIT_SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Dataset split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

num_eval_queries = min(MAX_EVAL_QUERIES, len(test_df))
print("Number of eval queries:", num_eval_queries)

Dataset split: train=32087, val=4011, test=4011
Number of eval queries: 500


## Train the models


### Tune the TF-IDF hyperparameters


In [7]:
def make_vectorizer(word_ngrams, char_ngrams, min_df, max_features, stop_words):
    word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=word_ngrams, min_df=min_df,
                                      max_features=max_features, sublinear_tf=True, stop_words=stop_words)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=char_ngrams, min_df=min_df,
                                      max_features=max_features, sublinear_tf=True)
    return FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])

def validation_precision_at_5(vectorizer, k=5):
    train_vectors = vectorizer.fit_transform(train_df["text_all"])
    query_vectors = vectorizer.transform(val_df["text_all"])
    similarity_matrix = cosine_similarity(query_vectors, train_vectors)
    hits = 0

    for query_index in range(len(val_df)):
        top_indices = np.argsort(-similarity_matrix[query_index])[:k]
        query_tags = set(val_df["domain_tag_list"].iloc[query_index])
        hits += sum(1 for catalog_index in top_indices
                    if set(train_df["domain_tag_list"].iloc[catalog_index]) & query_tags)

    return hits / (k * len(val_df))

tfidf_param_grid = [
    {"word_ngrams": (1, 1), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 3, "max_features": 50000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": "english"},
]

best_val_precision, best_tfidf_params = -1.0, None

for params in tfidf_param_grid:
    val_precision = validation_precision_at_5(make_vectorizer(**params))
    print(f"grid {params} -> val precision@5 = {val_precision:.3f}")
    if val_precision > best_val_precision:
        best_val_precision, best_tfidf_params = val_precision, params
print(f"best TF-IDF params (val precision@5={best_val_precision:.3f}): {best_tfidf_params}")


grid {'word_ngrams': (1, 1), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.627
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.619
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 3, 'max_features': 50000, 'stop_words': None} -> val precision@5 = 0.617
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': 'english'} -> val precision@5 = 0.625
best TF-IDF params (val precision@5=0.627): {'word_ngrams': (1, 1), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None}


### Vectorize the questions with TF-IDF


In [8]:
vectorizer = make_vectorizer(**best_tfidf_params)
tfidf_train_matrix = vectorizer.fit_transform(train_df["text_all"])
tfidf_val_matrix = vectorizer.transform(val_df["text_all"])
tfidf_test_matrix = vectorizer.transform(test_df["text_all"])

print(f"TF-IDF final (best params): train {tfidf_train_matrix.shape}, "
      f"val {tfidf_val_matrix.shape}, test {tfidf_test_matrix.shape}")

TF-IDF final (best params): train (32087, 125157), val (4011, 125157), test (4011, 125157)


### Generate the sentence embeddings


In [9]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

train_embeddings = embedding_model.encode(train_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
test_embeddings = embedding_model.encode(test_df.iloc[:num_eval_queries]["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
val_embeddings = embedding_model.encode(val_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
all_embeddings = embedding_model.encode(questions_df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)


Batches: 100%|██████████| 1254/1254 [00:46<00:00, 27.08it/s]


## Evaluate the models


### Evaluate TF-IDF on the test set


In [10]:
def evaluate_similarity(similarity_matrix, query_rows, catalog_rows, k):
    query_tag_lists = query_rows["domain_tag_list"].tolist()
    catalog_tag_lists = catalog_rows["domain_tag_list"].tolist()
    query_primary_tags = [tags.split("|")[0] for tags in query_rows["tags"]]
    catalog_primary_tags = [tags.split("|")[0] for tags in catalog_rows["tags"]]
    precision_scores, hit_rates, reciprocal_ranks, primary_precision_scores, random_baselines = [], [], [], [], []

    for query_index in range(len(query_rows)):
        top_indices = np.argsort(-similarity_matrix[query_index])[:k]
        query_tags = set(query_tag_lists[query_index])
        relevant_flags = [bool(set(catalog_tag_lists[catalog_index]) & query_tags) for catalog_index in top_indices]
        precision_scores.append(np.mean(relevant_flags))
        hit_rates.append(any(relevant_flags))
        for rank, is_relevant in enumerate(relevant_flags, start=1):
            if is_relevant:
                reciprocal_ranks.append(1 / rank)
                break
        else:
            reciprocal_ranks.append(0.0)
        primary_precision_scores.append(
            np.mean([catalog_primary_tags[catalog_index] == query_primary_tags[query_index]
                     for catalog_index in top_indices]))
        num_relevant_in_catalog = sum(1 for catalog_tags in catalog_tag_lists if set(catalog_tags) & query_tags)
        random_baselines.append(num_relevant_in_catalog / len(catalog_tag_lists))

    return (float(np.mean(precision_scores)), float(np.mean(hit_rates)), float(np.mean(reciprocal_ranks)),
            float(np.mean(primary_precision_scores)), float(np.mean(random_baselines)))

MODEL_RESULTS = {}

tfidf_similarity_matrix = cosine_similarity(tfidf_test_matrix[:num_eval_queries], tfidf_train_matrix)

for k in TOP_K:
    precision, hit_rate, mrr, primary_precision, random_baseline = evaluate_similarity(
        tfidf_similarity_matrix, test_df.iloc[:num_eval_queries], train_df, k)
    MODEL_RESULTS.setdefault("tfidf", {})[k] = (precision, hit_rate, mrr, primary_precision, random_baseline)

    print(f"TF-IDF      @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
          f"primary={primary_precision:.3f} (random={random_baseline:.3f})")

train_tag_lists = train_df["domain_tag_list"].tolist()
for query_index in range(min(3, num_eval_queries)):
    top_indices = np.argsort(-tfidf_similarity_matrix[query_index])[:5]
    print(f"QUERY: {test_df['title'].iloc[query_index][:90]}")
    for catalog_index in top_indices:
        print(f"  {tfidf_similarity_matrix[query_index, catalog_index]:.3f}  "
              f"{train_df['title'].iloc[catalog_index][:80]}  [{', '.join(train_tag_lists[catalog_index])}]")


TF-IDF      @5: precision=0.626 hit=0.898 MRR=0.755 primary=0.497 (random=0.075)
TF-IDF      @10: precision=0.604 hit=0.946 MRR=0.762 primary=0.478 (random=0.075)
QUERY: How can I make the dark black frames in the image white?
  0.386  How to change the background to white using OpenCV  [opencv]
  0.324  How to get text in white color from image in grey background and paste it on ano  [image-processing, numpy, opencv]
  0.315  I can't put white background on transparent image with OpenCV  [image-processing, opencv]
  0.304  smoothing or deleting white background in an image with python-opencv  [image-processing, opencv]
  0.297  How do I fill the image with white?  [image-processing, opencv]
QUERY: How to get a variable importance graph from a random forest using Tidymodels and vip
  0.300  Is there a way to get predictors importance from a neural network created on Tid  [machine-learning, neural-network]
  0.296  How do I use predictions from another model as feature in a tidymodels r

### Evaluate the sentence embeddings


In [11]:
embedding_similarity_matrix = cosine_similarity(test_embeddings, train_embeddings)

for k in TOP_K:
    precision, hit_rate, mrr, primary_precision, random_baseline = evaluate_similarity(
        embedding_similarity_matrix, test_df.iloc[:num_eval_queries], train_df, k)
    MODEL_RESULTS.setdefault("embeddings", {})[k] = (precision, hit_rate, mrr, primary_precision, random_baseline)
    print(f"embeddings  @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
          f"primary={primary_precision:.3f} (random={random_baseline:.3f})")


embeddings  @5: precision=0.695 hit=0.904 MRR=0.819 primary=0.520 (random=0.075)
embeddings  @10: precision=0.677 hit=0.938 MRR=0.824 primary=0.514 (random=0.075)


### Compare the models and choose the best one


In [12]:
print("=== model comparison (test split) ===")
for model_name, model_scores in MODEL_RESULTS.items():
    for k in sorted(model_scores):
        precision, hit_rate, mrr, primary_precision, random_baseline = model_scores[k]
        print(f"{model_name:10s} @{k}: precision={precision:.3f} hit={hit_rate:.3f} MRR={mrr:.3f} "
              f"primary={primary_precision:.3f} (random={random_baseline:.3f})")
best_model_name = max(MODEL_RESULTS, key=lambda model_name: MODEL_RESULTS[model_name][TOP_K[0]][0])
print(f"best model by precision@{TOP_K[0]}: {best_model_name}")


=== model comparison (test split) ===
tfidf      @5: precision=0.626 hit=0.898 MRR=0.755 primary=0.497 (random=0.075)
tfidf      @10: precision=0.604 hit=0.946 MRR=0.762 primary=0.478 (random=0.075)
embeddings @5: precision=0.695 hit=0.904 MRR=0.819 primary=0.520 (random=0.075)
embeddings @10: precision=0.677 hit=0.938 MRR=0.824 primary=0.514 (random=0.075)
best model by precision@5: embeddings


## Save the model artifacts for the app


In [13]:
output_dir = CONFIG["output_dir"]
save_npz(f"{output_dir}/X_train.npz", tfidf_train_matrix)
save_npz(f"{output_dir}/X_val.npz", tfidf_val_matrix)
save_npz(f"{output_dir}/X_test.npz", tfidf_test_matrix)
np.savez_compressed(f"{output_dir}/Y.npz", Y=label_matrix)
json.dump({"tag2idx": tag_to_index}, open(f"{output_dir}/tag_vocab.json", "w"), indent=1)
joblib.dump(vectorizer, f"{output_dir}/vectorizer.joblib")
np.savez_compressed(f"{output_dir}/embeddings_all.npz", E=all_embeddings)
json.dump({"emb_model": EMBEDDING_MODEL,
           "tfidf_params": {key: (list(value) if isinstance(value, tuple) else value)
                            for key, value in best_tfidf_params.items()},
           "model_results": {model_name: {str(k): list(scores) for k, scores in model_scores.items()}
                             for model_name, model_scores in MODEL_RESULTS.items()}},
          open(f"{output_dir}/model_selection.json", "w"), indent=1)
print(f"saved model artifacts to {output_dir}/")


saved model artifacts to outputs/
